Step 1: Import Required Libraries

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

Step 2: Spark Session

In [2]:
spark

NameError: name 'spark' is not defined

Step 3: Read CSV File

In [0]:
df = spark.read.csv(
    "/Volumes/workspace/default/online_retail_ll/online_retail_II.csv",
    header=True,
    inferSchema=True
)

Step 4: Check Schema

In [0]:
df = spark.read.csv(
    "/Volumes/workspace/default/online_retail_ll/online_retail_II.csv",
    header=True,
    inferSchema=True
)
df.printSchema()

root
 |-- Invoice: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- InvoiceDate: timestamp (nullable = true)
 |-- Price: double (nullable = true)
 |-- Customer ID: double (nullable = true)
 |-- Country: string (nullable = true)



Step 5: Number of Row

In [0]:
print("Rows :", df.count())

Rows : 1067371


Step 6: Number of Columns

In [0]:
print("Columns :", len(df.columns))

Columns : 8


Step 7: Display Columns

In [0]:
print(df.columns)

['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']


Step 10: Summary Statistics

In [0]:
df.describe().show()

+-------+-----------------+------------------+--------------------+-----------------+------------------+------------------+-----------+
|summary|          Invoice|         StockCode|         Description|         Quantity|             Price|       Customer ID|    Country|
+-------+-----------------+------------------+--------------------+-----------------+------------------+------------------+-----------+
|  count|          1067371|           1067371|             1062989|          1067371|           1067371|            824364|    1067371|
|   mean|537608.1499316233|28350.201592689715|            21848.25|  9.9388984711033| 4.649387727416074| 15324.63850435002|       NULL|
| stddev|26662.45044690487|17968.479697262945|   922.9197780233488|172.7057940767533|123.55305872146253|1697.4644503793093|       NULL|
|    min|           489434|             10002|  DOORMAT UNION J...|           -80995|         -53594.36|           12346.0|  Australia|
|    max|          C581569|                 m|  

Q1: Explain the roles of the Driver, Cluster Manager, and Executor in a Spark application. 

Ans:

**Driver**

- Creates the SparkSession.
- Converts user code into tasks.
- Builds the DAG (Directed Acyclic Graph).
- Schedules tasks and collects results.

**Cluster Manager**

- Allocates CPU, memory, and other resources.
- Manages executors.
- Examples: Standalone, YARN, Kubernetes.

**Executors**

- Execute tasks assigned by the Driver.
- Process data in parallel.
- Cache data and return results to the Driver.

Q2: How does Spark’s Lazy Evaluation strategy improve performance when chain-processing large datasets? 

Ans:

Spark records all transformations without executing them immediately. These transformations form a Lineage Graph (DAG). Execution begins only when an Action such as show() or count() is called.

**Advantages**

- Eliminates unnecessary computations.
- Optimizes execution plans.
- Reduces disk I/O.
- Improves performance.

Q3: Write a Spark command to read a CSV file located at "data/source.csv", ensuring the first row is treated as a header and inferSchema is enabled. 

Ans:


In [0]:
df = spark.read.csv(
    "data/source.csv",
    header=True,
    inferSchema=True
)

df.show(5)


---------------------------------------------------------------------------
IllegalArgumentException                  Traceback (most recent call last)
File <command-6096737571290943>, line 7
      1 df = spark.read.csv(
      2     "data/source.csv",
      3     header=True,
      4     inferSchema=True
      5 )
----> 7 df.show(5)

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/dataframe.py:1156, in DataFrame.show(self, n, truncate, vertical)
   1155 def show(self, n: int = 20, truncate: Union[bool, int] = True, vertical: bool = False) -> None:
-> 1156     print(self._show_string(n, truncate, vertical))

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/dataframe.py:909, in DataFrame._show_string(self, n, truncate, vertical)
    892     except ValueError:
    893         raise PySparkTypeError(
    894             errorClass="NOT_BOOL",
    895             messageParameters={
   (...)
    898             },
    899         )
    901 ta

Q4: What is the difference between CSV and Parquet in terms of storage (row-based vs. columnar) and why does it matter for performance? 

Ans:

**CSV**	                      ** Parquet**
- Row-based	                 Column-based
- Large file size      	     Compressed
- Slower	                 Faster
- Reads all columns	         Reads required columns only
- No Predicate Pushdown	     Supports Predicate Pushdown

**Why?**

Parquet improves performance because Spark reads only the required columns.

Q5: Given a DataFrame df, write a query to select the columns product_id and price where the category is 'Electronics'. 

In [0]:
from pyspark.sql.functions import col

# Create sample DataFrame for the question
data = [
    (1, "Electronics", 299.99),
    (2, "Clothing", 49.99),
    (3, "Electronics", 599.99),
    (4, "Electronics", 149.99),
    (5, "Home", 89.99)
]

df = spark.createDataFrame(data, ["product_id", "category", "price"])

# Filter and select as per Question 5
df.filter(
    col("category")=="Electronics"
).select(
    "product_id",
    "price"
).show()

+----------+------+
|product_id| price|
+----------+------+
|         1|299.99|
|         3|599.99|
|         4|149.99|
+----------+------+



Q6: Write the code to "revise" a DataFrame by renaming the column old_name to new_name and casting the price column from a String to a Double. 

In [0]:
from pyspark.sql.functions import col

df = df.withColumnRenamed(
    "old_name",
    "new_name"
)

df = df.withColumn(
    "price",
    col("price").cast("double")
)

df.printSchema()

root
 |-- product_id: long (nullable = true)
 |-- category: string (nullable = true)
 |-- price: double (nullable = true)



**Q7: How does Spark use the Lineage Graph (DAG) to provide fault tolerance if a worker node fails? **

**Answer:**

Spark stores every transformation in a Lineage Graph (DAG).

If an executor fails, Spark does not reload the entire dataset. It recomputes only the lost partitions using the lineage information.

**Benefits:**

- Automatic recovery
- Fault tolerance
- Efficient recomputation

Q8: Write a query to filter a DataFrame df_orders for rows where the status is 'Completed' AND the amount is greater than 1000. 

In [0]:
from pyspark.sql.functions import col

# Create sample DataFrame for the question
data = [
    (1, "Completed", 1200),
    (2, "Pending", 800),
    (3, "Completed", 1500),
    (4, "Completed", 900),
    (5, "Cancelled", 1100)
]

df_orders = spark.createDataFrame(data, ["order_id", "status", "amount"])

# Filter as per Question 8
df_orders.filter(
    (col("status")=="Completed") &
    (col("amount")>1000)
).show()

+--------+---------+------+
|order_id|   status|amount|
+--------+---------+------+
|       1|Completed|  1200|
|       3|Completed|  1500|
+--------+---------+------+



**Q9: Explain the concept of Predicate Pushdown in Parquet and how it affects the amount of data loaded into memory. **

**Answer:**

Predicate Pushdown pushes filter conditions to the storage layer (such as Parquet).

**Example:**

Instead of reading all rows,

Spark reads only matching rows.

**Benefits:**

- Faster execution
- Lower memory usage
- Less disk I/O

Q10: Write a code snippet to add a new column final_price which is the base_price multiplied by 1.18 (18% tax). 

In [0]:
from pyspark.sql.functions import col

df = df.withColumn(
    "final_price",
    col("price")*1.18
)

df.show()

+----------+-----------+------+-----------+
|product_id|   category| price|final_price|
+----------+-----------+------+-----------+
|         1|Electronics|299.99|   353.9882|
|         2|   Clothing| 49.99|    58.9882|
|         3|Electronics|599.99|   707.9882|
|         4|Electronics|149.99|   176.9882|
|         5|       Home| 89.99|   106.1882|
+----------+-----------+------+-----------+



Q11: What is the difference between Transformations and Actions? Provide two examples of each. 

Ans:
  
**Transformations	**         **Actions**
Lazy	                 Immediate execution
Return DataFrame	     Return result
Build DAG	             Execute DAG

Examples of Transformations

In [0]:
df.filter(col("price")>100)

df.select("product")

Examples of `Actions`

In [0]:
df.show()

df.count()

+----------+-----------+------+-----------+
|product_id|   category| price|final_price|
+----------+-----------+------+-----------+
|         1|Electronics|299.99|   353.9882|
|         2|   Clothing| 49.99|    58.9882|
|         3|Electronics|599.99|   707.9882|
|         4|Electronics|149.99|   176.9882|
|         5|       Home| 89.99|   106.1882|
+----------+-----------+------+-----------+



5

Q12: Write the Spark command to load a Parquet file from "path/to/input", filter out any rows where user_id is null, and save the result as a CSV at "path/to/output". 

In [0]:
from pyspark.sql.functions import col

# Create sample data to demonstrate the pattern
data = [(1, "Alice"), (2, "Bob"), (None, "Charlie"), (4, "Diana")]
df = spark.createDataFrame(data, ["user_id", "name"])

# Filter out rows where user_id is null
filtered_df = df.filter(col("user_id").isNotNull())

# Show the result
filtered_df.show()

# To write to CSV (commented out - replace with actual path):
# filtered_df.write.mode("overwrite").option("header", True).csv("/Volumes/catalog/schema/volume/output")

+-------+-----+
|user_id| name|
+-------+-----+
|      1|Alice|
|      2|  Bob|
|      4|Diana|
+-------+-----+



Q13: In Spark Architecture, what is the difference between Client Mode and Cluster Mode? 

Ans:

**Client Mode	    **                        **Cluster Mode**
- Driver runs on local machine	        Driver runs inside cluster
- Development                   	        Production
- Client must remain connected	        Runs independently

Q14: Write a query to filter a dataset for rows where the region is 'North' OR the priority is 'High'. 

In [0]:
from pyspark.sql.functions import col

# Create sample DataFrame for the question
data = [
    (1, "North", "Medium"),
    (2, "South", "High"),
    (3, "North", "Low"),
    (4, "East", "Medium"),
    (5, "West", "High")
]

df_filter = spark.createDataFrame(data, ["id", "region", "priority"])

# Filter as per Question 14: region is 'North' OR priority is 'High'
df_filter.filter(
    (col("region")=="North") |
    (col("priority")=="High")
).show()

+---+------+--------+
| id|region|priority|
+---+------+--------+
|  1| North|  Medium|
|  2| South|    High|
|  3| North|     Low|
|  5|  West|    High|
+---+------+--------+



Q15: When exploring a dataset, why is it safer to use .show(5) instead of .collect() on a multi-terabyte dataset? 

**Answer:**

show(5) returns only five rows.

collect() returns the entire dataset to the Driver.

**For large datasets:**

- collect() may cause OutOfMemoryError.
- Slows execution.
- Can crash the Driver.

Therefore, show(5) is safer for data exploration.

**# Brief Insights**

1. Spark Architecture enables distributed processing using the Driver, Cluster Manager, and Executors for efficient execution.

2. Lazy Evaluation and DAG optimize query execution by delaying computation until an action is performed.

3. DataFrame transformations, filtering, schema handling, and null value processing improve data quality and efficiency.

4. Parquet provides better performance than CSV due to its columnar storage, compression, and Predicate Pushdown.

5. Building an ETL pipeline (Read → Transform → Filter → Write) demonstrates efficient data processing using PySpark.
